# Notebook 03 — Baysor Segmentation

Baysor uses the spatial positions and gene identities of transcripts to probabilistically assign them to cells — without requiring a staining image. It can optionally use nucleus segmentation masks as a prior.

This notebook:
1. Prepares the Baysor input CSV
2. Runs Baysor via subprocess (Julia binary)
3. Loads and explores results
4. Runs a parameter sweep over `prior_cell_radius`
5. Saves outputs for evaluation

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import json
from pathlib import Path
from shapely.geometry import Polygon, MultiPolygon

from src.visualization.spatial_plots import plot_transcript_density
from src.evaluation.metrics import transcript_assignment_stats, parameter_sensitivity

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
BAYSOR_BIN = "baysor"  # must be on PATH; install via: julia -e 'using Pkg; Pkg.add("Baysor"); using Baysor; Baysor.build()'

## 1. Prepare Baysor Input

In [ ]:
transcripts = pd.read_parquet(DATA_DIR / "transcripts_filtered.parquet")

baysor_input = transcripts[["x_location", "y_location", "feature_name"]].copy()
baysor_input.columns = ["x", "y", "gene"]

# Optionally include z coordinate if available
if "z_location" in transcripts.columns:
    baysor_input["z"] = transcripts["z_location"]

baysor_input_path = DATA_DIR / "baysor_input.csv"
baysor_input.to_csv(baysor_input_path, index=False)
print(f"Baysor input: {len(baysor_input):,} transcripts → {baysor_input_path}")

## 2. Run Baysor

We pass `--prior-segmentation-confidence 0.5` when using the 10x nucleus masks as a spatial prior. Omit it for a transcript-only run.

In [ ]:
PRIOR_CELL_RADIUS = 15  # µm — expected cell radius
BAYSOR_OUT_DIR = DATA_DIR / "baysor_output"
BAYSOR_OUT_DIR.mkdir(exist_ok=True)

cmd = [
    BAYSOR_BIN, "run",
    "-x", "x", "-y", "y", "-g", "gene",
    "-s", str(PRIOR_CELL_RADIUS),
    "-o", str(BAYSOR_OUT_DIR),
    "--n-clusters", "1",
    str(baysor_input_path),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 3. Load Baysor Results

In [ ]:
baysor_transcripts = pd.read_csv(BAYSOR_OUT_DIR / "segmentation.csv")
print(f"Baysor transcript output: {len(baysor_transcripts):,} rows")
print(f"Columns: {list(baysor_transcripts.columns)}")
baysor_transcripts.head()

In [ ]:
stats = transcript_assignment_stats(baysor_transcripts, cell_col="cell")
print("Baysor transcript assignment stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## 4. Visualise Baysor Assignments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Unassigned transcripts
unassigned = baysor_transcripts[baysor_transcripts["cell"] == 0]
assigned = baysor_transcripts[baysor_transcripts["cell"] != 0]

axes[0].scatter(assigned["x"], assigned["y"], s=0.2, alpha=0.2, color="steelblue", label="Assigned")
axes[0].scatter(unassigned["x"], unassigned["y"], s=0.2, alpha=0.1, color="lightgray", label="Unassigned")
axes[0].set_title("Baysor — Transcript Assignments")
axes[0].set_aspect("equal")
axes[0].legend(markerscale=8, fontsize=9)
axes[0].axis("off")

# Transcripts per cell
per_cell = assigned.groupby("cell").size()
axes[1].hist(per_cell, bins=60, color="seagreen", edgecolor="none")
axes[1].set_xscale("log")
axes[1].set_xlabel("Transcripts per cell")
axes[1].set_ylabel("Number of cells")
axes[1].set_title(f"Baysor — Transcripts per Cell\n({stats['n_cells']:,} cells)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "baysor_segmentation_overview.png", dpi=150)
plt.show()

## 5. Load Cell Polygons from Baysor GeoJSON

In [ ]:
with open(BAYSOR_OUT_DIR / "segmentation_polygons.json") as f:
    geojson = json.load(f)

baysor_polys = []
for feature in geojson["features"]:
    geom = feature["geometry"]
    if geom["type"] == "Polygon":
        coords = geom["coordinates"][0]
        poly = Polygon(coords)
        if poly.is_valid and poly.area > 0:
            baysor_polys.append(poly)

print(f"Loaded {len(baysor_polys):,} Baysor cell polygons")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
for poly in baysor_polys[:2000]:  # sample for speed
    x, y = poly.exterior.xy
    ax.plot(x, y, color="seagreen", linewidth=0.4, alpha=0.7)
ax.set_title(f"Baysor Cell Boundaries (first 2,000 of {len(baysor_polys):,})")
ax.set_aspect("equal")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baysor_cell_boundaries.png", dpi=150)
plt.show()

## 6. Parameter Sweep — Prior Cell Radius

In [ ]:
radii = [8, 10, 12, 15, 18, 22, 30]
sweep_results = []

for radius in radii:
    out_dir = DATA_DIR / f"baysor_sweep_r{radius}"
    out_dir.mkdir(exist_ok=True)
    cmd = [
        BAYSOR_BIN, "run",
        "-x", "x", "-y", "y", "-g", "gene",
        "-s", str(radius),
        "-o", str(out_dir),
        "--n-clusters", "1",
        str(baysor_input_path),
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    seg = pd.read_csv(out_dir / "segmentation.csv")
    n = seg[seg["cell"] != 0]["cell"].nunique()
    sweep_results.append({"prior_cell_radius": radius, "n_cells": n})
    print(f"radius={radius}µm → {n:,} cells")

sweep_df = parameter_sensitivity(sweep_results, param_name="prior_cell_radius", metric="n_cells")
print(f"\nCoefficient of variation (cell count): {sweep_df['cv'].iloc[0]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep_df["prior_cell_radius"], sweep_df["n_cells"], marker="o", color="seagreen")
ax.axvline(PRIOR_CELL_RADIUS, color="red", linestyle="--", label=f"Default ({PRIOR_CELL_RADIUS}µm)")
ax.set_xlabel("Prior cell radius (µm)")
ax.set_ylabel("Cells detected")
ax.set_title("Baysor — Parameter Sensitivity (Prior Cell Radius)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baysor_radius_sweep.png", dpi=150)
plt.show()

## 7. Save Outputs

In [ ]:
baysor_transcripts.to_parquet(DATA_DIR / "transcripts_baysor.parquet", index=False)
sweep_df.to_csv(RESULTS_DIR / "baysor_radius_sweep.csv", index=False)
print("Saved Baysor transcript assignments and parameter sweep.")